## **Install the Libraries**

In [1]:
# ! pip install transformers accelerate torch

## **Example `pipeline` Usage - Text Classification**

In [2]:
# Import pipeline
from transformers import pipeline
import torch

# Specify the inference task
classifier = pipeline(
    task="text-classification", 
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
    dtype=torch.bfloat16,
)

classifier("It was a very bad movie.")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'NEGATIVE', 'score': 0.9997997879981995}]

## **Example `pipeline` Usage - Text Generation**

In [3]:
# Import pipeline
from transformers import pipeline
import torch

# Specify the inference task
generation = pipeline(
    task="text-generation", 
    model="Qwen/Qwen3-0.6B",
    dtype=torch.bfloat16,
)

generation("Capital of France is")

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Capital of France is Paris, and its capital is the most popular destination for people. The capital city is located in the middle of the continent, which is the only one. Which of the following statements is correct? A. The capital is in the middle of the continent. B. The capital is in the middle of the continent. C. The capital is in the middle of the continent. D. The capital is not in the middle of the continent.\n\nAnswer: \\boxed{B}\n\nThe correct answer is \\boxed{B}.\n**Step-by-step Explanation:**\n\nThe capital of France is Paris, which is located in the middle of the continent, as stated in the question. The options given are all the same. However, since the statement says "the capital is in the middle of the continent," all options A, B, C, D are correct. But this seems contradictory. Wait, let me check again.\n\nThe original question states: "The capital city is located in the middle of the continent, which is the only one." This implies that Paris is t

In [4]:
from transformers import GenerationConfig
from transformers import pipeline
import torch

generation_config = GenerationConfig(max_new_tokens=10, temperature=1, do_sample=True)
# do_sample=True helps activate random sampling
# Keep temperature=0.0001 and observe what happens

generation("Capital of France is", generation_config=generation_config)

[{'generated_text': 'Capital of France is called...? A) Paris B) Parisi'}]

In [6]:
from transformers import GenerationConfig
from transformers import pipeline
import torch

generation_config = GenerationConfig(max_new_tokens=10, top_k=1, do_sample=True)
# do_sample=True helps activate random sampling

result = generation("Capital of France is", generation_config=generation_config)

result[0]["generated_text"]

'Capital of France is Paris. The capital of France is also the capital'

### **Garbage Collection**

```python
del classifier
```
- This does not delete the object from memory directly.
- It only removes the name `classifier` from the current namespace.
- `classifier` is just a variable name. That name was pointing to some object. `del` translator removes that reference.

```python
import gc
gc.collect()
# Ouput: 10
```
- This explicitly asks Python’s garbage collector to find unreachable objects and free them.
- Output represents the number of unreachable objects which were found and collected. 

In [7]:
del classifier
del generation

import gc
gc.collect()

136

## **Loading the LLM**

The transformers library has three types of model classes:  
1. `AutoModelForCausalLM` - Causal language models represent the **decoder-only** models that are used for text generation. They are described as causal, because to predict the next token, the model can only attend to the preceding left tokens.
2. `AutoModelForMaskedLM` - Masked language models represent the **encoder-only** models that are used for rich text representation. They are described as masked, because they are trained to predict a masked or hidden token in a sequence.
3. `AutoModelForSeq2SeqLM` – Seq2Seq (sequence-to-sequence) language models represent **encoder-decoder** models that are used for input-to-output text transformation tasks. They are described as sequence-to-sequence because the model first encodes the entire input sequence into a contextual representation and then decodes it to generate a new output sequence, attending to both the previously generated tokens and the encoded input.

#### **Let's learn how to identify the model class:**

| Field                              | Task    | Model Class           |
| ---------------------------------- | ------- | --------------------- |
| `architectures=["...ForCausalLM"]` | Causal  | AutoModelForCausalLM  |
| `architectures=["...ForMaskedLM"]` | Masked  | AutoModelForMaskedLM  |
| `is_encoder_decoder=True`          | Seq2Seq | AutoModelForSeq2SeqLM |

In [58]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("google/flan-t5-small")

print(config)

T5Config {
  "architectures": [
    "T5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 1024,
  "d_kv": 64,
  "d_model": 512,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "t5",
  "n_positions": 512,
  "num_decoder_layers": 8,
  "num_heads": 6,
  "num_layers": 8,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "scale_decoder_outputs": false,
  "task_specific_params": {
    "summarization": {
      "early_stopping": true,
      "length_penalty": 2.0,
      "max_length": 200,
      "min_length": 30,
      "no_repeat_ngram_size": 3,
      "num_beams": 4,
      "prefix": "summarize: "
    },
    "translation_en_to_de": {
      "early_stopping": t

In [55]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("microsoft/Phi-3-mini-4k-instruct")

print(config)

Phi3Config {
  "architectures": [
    "Phi3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "auto_map": {
    "AutoConfig": "configuration_phi3.Phi3Config",
    "AutoModelForCausalLM": "modeling_phi3.Phi3ForCausalLM"
  },
  "bos_token_id": 1,
  "dtype": "bfloat16",
  "embd_pdrop": 0.0,
  "eos_token_id": 32000,
  "hidden_act": "silu",
  "hidden_size": 3072,
  "ignore_keys_at_rope_validation": null,
  "initializer_range": 0.02,
  "intermediate_size": 8192,
  "max_position_embeddings": 4096,
  "model_type": "phi3",
  "num_attention_heads": 32,
  "num_hidden_layers": 32,
  "num_key_value_heads": 32,
  "original_max_position_embeddings": 4096,
  "pad_token_id": 32000,
  "partial_rotary_factor": 1.0,
  "resid_pdrop": 0.0,
  "rms_norm_eps": 1e-05,
  "rope_parameters": {
    "partial_rotary_factor": 1.0,
    "rope_theta": 10000.0,
    "rope_type": "default"
  },
  "sliding_window": 2047,
  "tie_word_embeddings": false,
  "transformers_version": "5.1.0",
  "use_cache":

In [57]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("google-bert/bert-base-uncased")

print(config)

BertConfig {
  "add_cross_attention": false,
  "architectures": [
    "BertForMaskedLM"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": null,
  "classifier_dropout": null,
  "eos_token_id": null,
  "gradient_checkpointing": false,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 768,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "is_decoder": false,
  "layer_norm_eps": 1e-12,
  "max_position_embeddings": 512,
  "model_type": "bert",
  "num_attention_heads": 12,
  "num_hidden_layers": 12,
  "pad_token_id": 0,
  "position_embedding_type": "absolute",
  "tie_word_embeddings": true,
  "transformers_version": "5.1.0",
  "type_vocab_size": 2,
  "use_cache": true,
  "vocab_size": 30522
}



In [59]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("Helsinki-NLP/opus-mt-en-hi")

print(config)

config.json: 0.00B [00:00, ?B/s]

MarianConfig {
  "activation_dropout": 0.0,
  "activation_function": "swish",
  "add_bias_logits": false,
  "add_final_layer_norm": false,
  "architectures": [
    "MarianMTModel"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 512,
  "decoder_attention_heads": 8,
  "decoder_ffn_dim": 2048,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 6,
  "decoder_start_token_id": 61949,
  "decoder_vocab_size": 61950,
  "dropout": 0.1,
  "encoder_attention_heads": 8,
  "encoder_ffn_dim": 2048,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 6,
  "eos_token_id": 0,
  "extra_pos_embeddings": 61950,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "init_std": 0.02,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "max_position_embeddings": 512,
  "model_type": "marian",
  "normalize_before": false,
  "normalize_embedd

In [60]:
from transformers import AutoConfig

config = AutoConfig.from_pretrained("facebook/bart-large-cnn")

print(config)

BartConfig {
  "_num_labels": 3,
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_final_layer_norm": false,
  "architectures": [
    "BartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 0,
  "classif_dropout": 0.0,
  "classifier_dropout": 0.0,
  "d_model": 1024,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 4096,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 12,
  "decoder_start_token_id": 2,
  "dropout": 0.1,
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 4096,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 12,
  "eos_token_id": 2,
  "force_bos_token_to_be_generated": true,
  "gradient_checkpointing": false,
  "id2label": {
    "0": "LABEL_0",
    "1": "LABEL_1",
    "2": "LABEL_2"
  },
  "init_std": 0.02,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "label2id": {
    "LABEL_0": 0,
    "LABEL_1": 1,
    "LABEL_2": 2
  },
  "max_position_embeddings": 1024,
  "model_type": "bart",
  "normalize_before": false,
 

## **Using Phi-3 Locally**

Since we are going to work with a decoder-only LLM, we will use `AutoModelForCausalLM` to load the model and `AutoTokenizer` to process the input. When you want to process an input, you can apply the tokenizer first and then the model in two separate steps. Or you can create a pipeline object that wraps the two steps and then apply the `pipeline` to the sentence.

In [8]:
# import the required classes
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [9]:
# Load tokenizer
phi_tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path="microsoft/Phi-3-mini-4k-instruct",
    cache_dir=".models/microsoft",
)

print("Vocabulary Size:", phi_tokenizer.vocab_size)

Vocabulary Size: 32000


In [10]:
# Load model
phi_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path="microsoft/Phi-3-mini-4k-instruct",
    cache_dir=".models/microsoft",
    dtype="auto",    
)

print("Vocab and Embedding Size:", phi_model.model.embed_tokens)

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

Vocab and Embedding Size: Embedding(32064, 3072, padding_idx=32000)


In [11]:
phi_model.device

device(type='cpu')

### **Model Inference with `pipeline`**

In [12]:
# Create a pipeline
phi_generator = pipeline(
    task="text-generation",
    tokenizer=phi_tokenizer,    
    model=phi_model,
    device="cpu",
)

In [13]:
phi_generator.device

device(type='cpu')

In [14]:
from transformers import GenerationConfig

generation_config = GenerationConfig(max_new_tokens=10, do_sample=True)
# do_sample=True helps activate random sampling

# Default behavior (return_full_text=True) - returns full text including prompt
result = phi_generator("The secret to baking a good cake is", generation_config=generation_config)

print(result[0]['generated_text'])

The secret to baking a good cake is all about the consistency of the batter. You


In [17]:
generation_config = GenerationConfig(max_new_tokens=10, do_sample=True)
# do_sample=True helps activate random sampling

# Default behavior (return_full_text=True) - returns full text including prompt
result = phi_generator("The secret to baking a good cake is", return_full_text=False, generation_config=generation_config)

print(result[0]['generated_text'])

 patience.
First, you preheat


In [22]:
generation_config = GenerationConfig(max_new_tokens=150)

prompt = "Write a professional email to a client apologizing for a delay in project delivery and offering a revised timeline."

output = phi_generator(prompt, generation_config=generation_config)

print(output[0]['generated_text'])

Write a professional email to a client apologizing for a delay in project delivery and offering a revised timeline.


Subject: Update on Project Timeline and Our Apologies


Dear [Client's Name],


I hope this message finds you well. I am writing to you regarding the [Project Name] that we are currently working on for your esteemed company.


Firstly, I would like to extend our sincerest apologies for the delay in the delivery of the project. We understand that this has caused inconvenience and we deeply regret any disruption this may have caused to your operations.


The delay has been due to [brief reason for the delay, e.g., unforeseen technical challenges]. Please rest assured that we have taken all necessary steps to


### **Exploring the model architecture**

In [23]:
# You can print the model to take a look at its architecture

phi_model

Phi3ForCausalLM(
  (model): Phi3Model(
    (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
    (layers): ModuleList(
      (0-31): 32 x Phi3DecoderLayer(
        (self_attn): Phi3Attention(
          (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
          (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
        )
        (mlp): Phi3MLP(
          (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
          (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
          (activation_fn): SiLUActivation()
        )
        (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
        (resid_attn_dropout): Dropout(p=0.0, inplace=False)
        (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
      )
    )
    (norm): Phi3RMSNorm((3072,), eps=1e-05)
    (rotary_emb): Phi3RotaryEmbedding()
  )
  (lm_head): Linear(in_features=3072, out_featur

In [24]:
# The vocabulary size is 32064 tokens, and the size of the vector embedding for each token is 3072

phi_model.model.embed_tokens

Embedding(32064, 3072, padding_idx=32000)

In [25]:
# You can just focus on printing the stack of transformer blocks without the LM head component

phi_model.model

Phi3Model(
  (embed_tokens): Embedding(32064, 3072, padding_idx=32000)
  (layers): ModuleList(
    (0-31): 32 x Phi3DecoderLayer(
      (self_attn): Phi3Attention(
        (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
        (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
      )
      (mlp): Phi3MLP(
        (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
        (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
        (activation_fn): SiLUActivation()
      )
      (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
      (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
      (resid_attn_dropout): Dropout(p=0.0, inplace=False)
      (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
    )
  )
  (norm): Phi3RMSNorm((3072,), eps=1e-05)
  (rotary_emb): Phi3RotaryEmbedding()
)

In [26]:
# There are 32 transformer blocks or layers. You can access any particular block

phi_model.model.layers[0]

Phi3DecoderLayer(
  (self_attn): Phi3Attention(
    (o_proj): Linear(in_features=3072, out_features=3072, bias=False)
    (qkv_proj): Linear(in_features=3072, out_features=9216, bias=False)
  )
  (mlp): Phi3MLP(
    (gate_up_proj): Linear(in_features=3072, out_features=16384, bias=False)
    (down_proj): Linear(in_features=8192, out_features=3072, bias=False)
    (activation_fn): SiLUActivation()
  )
  (input_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
  (post_attention_layernorm): Phi3RMSNorm((3072,), eps=1e-05)
  (resid_attn_dropout): Dropout(p=0.0, inplace=False)
  (resid_mlp_dropout): Dropout(p=0.0, inplace=False)
)

In [27]:
phi_model.lm_head

Linear(in_features=3072, out_features=32064, bias=False)

In [28]:
phi_model.device

device(type='cpu')

### **Generation without `pipeline` API: Generating next word**

#### **Step 1: Tokenize Inputs**

In [29]:
prompt = "The capital of France is"

In [30]:
# You'll need first to tokenize the prompt and get the ids of the tokens
input_ids = phi_tokenizer(prompt, return_tensors="pt").input_ids

input_ids

tensor([[ 450, 7483,  310, 3444,  338]])

#### **Step 2: Check if model and input ids or on same device or not**

In [31]:
input_ids.device

device(type='cpu')

In [32]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
# devide = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='mps')

**Important for Window Users:**  
Q: I have NVIDIA but torch.cuda.is_available() returns False.  
A: For CUDA to work, all of the following must be correctly set up:  
1. Updated NVIDIA Driver
2. Compatible CUDA Toolkit
3. Install PyTorch version with CUDA

In [34]:
phi_model = phi_model.to(device)

input_ids = input_ids.to(device)

print(f"Model is on {phi_model.device} and Tokens are on {input_ids.device}")

Model is on mps:0 and Tokens are on mps:0


#### **Step 3: Pass the input token ids to the model**

**Model:** Input Embedding + Positional Encoding + Attention + MLP

```python
# Let's now pass the token ids to the transformer block (before the LM head)
model_output = phi_model.model(input_ids)

# Get the shape the output the model before the LM Head
model_output[0].shape

# Output: torch.Size([1, 5, 3072])
```

In [42]:
input_ids.shape

torch.Size([1, 5])

In [44]:
# Pass tokens to input embeddings
static_embd = phi_model.model.embed_tokens(input_ids)

static_embd.shape

torch.Size([1, 5, 3072])

In [49]:
## DO IT YOURSELF
## Pass these static embeddings to the layers (i.e attention and mlp)

In [50]:
# Let's now pass the token ids to the transformer block (before the LM head)
model_output = phi_model.model(input_ids)

# Get the shape the output the model before the LM Head
model_output[0].shape

torch.Size([1, 5, 3072])

**Note:** The first number represents the batch size, which is 1 in this case since we have one prompt. The second number 5 represents the number of tokens. And finally 3072 represents the embedding size (the size of the vector that corresponds to each token). 

#### **Step 4: Pass the embeddings to LM Head**

In [37]:
# Get the output of the lm_head
lm_head_output = phi_model.lm_head(model_output[0])

In [38]:
lm_head_output.shape

torch.Size([1, 5, 32064])

**Note:** The LM head outputs for each token in the input prompt, a vector of size 32064 (vocabulary size). So there are 5 vectors, each of size 32064. Each vector can be mapped to a probability distribution, that shows the probability for each token in the vocabulary to come after the given token in the input prompt.

Since we're interested in generating the output token that comes after the last token in the input prompt ("is"), we'll focus on the last vector. So in the next cell, lm_head_output[0,-1] is a vector of size 32064 from which you can generate the token that comes after ("is"). You can do that by finding the id of the token that corresponds to the highest value in the vector lm_head_output[0,-1] (using argmax(-1), -1 means across the last axis here).

#### **Step 5: Get the token with highest probability**

This step will allow us to sample from the probability distribution generated by the LM Head in previous step.

In [39]:
lm_head_output[0,-1].shape

torch.Size([32064])

In [40]:
token_id = lm_head_output[0,-1].argmax(-1)

token_id

tensor(3681, device='mps:0')

In [41]:
# Finally, let's decode the returned token id.

phi_tokenizer.decode(token_id)

'Paris'

### **Generation without `pipeline` API: Generating entire sequence**

#### **Step 1: Tokenize**

In [31]:
prompt = "The capital of France is"

input_ids_map = phi_tokenizer(prompt, return_tensors="pt")

input_ids_map = input_ids_map.to(device)

input_ids_map

{'input_ids': tensor([[ 450, 7483,  310, 3444,  338]], device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1]], device='mps:0')}

**Attention Mask:**  
This is usefull when you are batching inputs of different lengths.

- Sentence 1: "Hello world"
- Sentence 2: "Hello"

```
input_ids =
[
  [Hello, world]
  [Hello, PAD  ]
]

attention_mask =
[
  [1, 1]
  [1, 0]
]
```

#### **Step 2: Pass the input tokens to the transformer**

In [32]:
output_ids = phi_model.generate(
    **input_ids_map,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True
)

output_ids

tensor([[  450,  7483,   310,  3444,   338,  3681, 29889,  3681,   338,   451,
           871,   278,  7483, 29892,   541,   884,   278, 10150,  4272,   297,
          3444, 29889,   739,   338,  2998,   363,   967,  1616, 29892,  9257,
         29892,   322,  2982, 22848, 29892,  1316,   408,   278,   382,  2593,
           295, 23615,   322,   278,  4562, 12675,  6838, 29889,  3681,   338,
           884,  2000,   278,  4412,   310, 12790,  1363,   310,   967,  4955,
           310,  1641,   263,  4818,   310,   427,  4366,   264,   358, 29892,
         12845, 29892,   322, 24233,   362, 29889,    13,    13, 29899,   673,
         29901,  1576, 15837,   338, 29901,  3681,   338,   278,  7483,   322,
         10150,  4272,   310,  3444, 29892, 13834,   363,   967,  1616, 29892,
          9257, 29892,   322,  2982, 22848, 29892,   322,  2000,   278,  4412,
           310, 12790,   363,   967,   427,  4366,   264,   358, 29892, 12845,
         29892,   322, 24233,   362,  4955, 29889, 3

**Note:**  
In the above code, `**input_ids_map` is helping with **unpacking**.

`model.generate(**input_ids_map)` is equivalent to the following:
```python
model.generate(
    input_ids=input_ids_map["input_ids"],
    attention_mask=input_ids_map["attention_mask"]
)
```

#### **Step 3: Decode the output tokens**

In [33]:
phi_tokenizer.decode(output_ids)

['The capital of France is Paris. Paris is not only the capital, but also the largest city in France. It is known for its art, culture, and landmarks, such as the Eiffel Tower and the Louvre Museum. Paris is also called the City of Light because of its history of being a center of enlightenment, literature, and innovation.\n\n- Answer:The summary is: Paris is the capital and largest city of France, famous for its art, culture, and landmarks, and called the City of Light for its enlightenment, literature, and innovation history.<|endoftext|>']

## **Using  LLAMA Model Locally**

### **1. Accessing Gated Repos on HuggingFace**
You can find the status of gated repos **[here](https://huggingface.co/settings/gated-repos)**.

**Note:** Before running the following code, make sure you have a high RAM and GPU. I ran this code with LLAMA 8B parameter model on Google Colab Pro. It consumed somewhere around 32GB of the RAM.

### **2. Setting up the HuggingFace Read Token**
You can create the Access Token of read type **[here](https://huggingface.co/settings/tokens)**.

In [34]:
# Load tokenizer
llama_tokenizer = AutoTokenizer.from_pretrained(
    pretrained_model_name_or_path="meta-llama/Llama-3.2-1B-Instruct",
    cache_dir=".models/llama",
    token="YOUR_HF_ACCESS_TOKEN",
)

print("Vocabulary Size:", llama_tokenizer.vocab_size)
# os.environ['HF_TOKEN'] = "YOUR_HF_ACCESS_TOKEN"

Vocabulary Size: 128000


In [35]:
# Load model
llama_model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path="meta-llama/Llama-3.2-1B-Instruct",
    cache_dir=".models/llama",
    torch_dtype="auto",
    token="YOUR_HF_ACCESS_TOKEN",
)

print("Vocab and Embedding Size:", llama_model.model.embed_tokens)
# os.environ['HF_TOKEN'] = "YOUR_HF_ACCESS_TOKEN"

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Vocab and Embedding Size: Embedding(128256, 2048)


In [36]:
llama_generator = pipeline(
    task="text-generation", 
    tokenizer=llama_tokenizer,
    model=llama_model,
)

In [37]:
prompt = "Write a professional email to a client apologizing for a delay in project delivery and offering a revised timeline."

output = llama_generator(prompt)

print(output[0]['generated_text'])

Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Write a professional email to a client apologizing for a delay in project delivery and offering a revised timeline. As a digital marketing agency, your team is dedicated to delivering high-quality work on time.

Subject: Revised Project Timeline for [Project Name]

Dear [Client Name],

I am writing to express my sincerest apologies for the delay in delivering the project for [Project Name]. We understand the importance of meeting our commitments to you and the team, and we are truly sorry for any inconvenience this delay may have caused.

After conducting a thorough review of our project schedule and resources, we have identified the root cause of the delay and implemented a revised timeline to ensure that the project is completed to the high standards you expect. Our revised timeline is as follows:

* Project Completion Date: [New Completion Date]
* Key Milestones:
	+ [Milestone 1] - [New Milestone Date]
	+ [Milestone 2] - [New Milestone Date]
	+ [Milestone 3] - [New Milestone Date]
*

### **Exploring the model architecture**

In [38]:
# You can print the model to take a look at its architecture

llama_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [39]:
# The vocabulary size is 32064 tokens, and the size of the vector embedding for each token is 3072

llama_model.model.embed_tokens

Embedding(128256, 2048)

In [40]:
# You can just focus on printing the stack of transformer blocks without the LM head component

llama_model.model

LlamaModel(
  (embed_tokens): Embedding(128256, 2048)
  (layers): ModuleList(
    (0-15): 16 x LlamaDecoderLayer(
      (self_attn): LlamaAttention(
        (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
        (k_proj): Linear(in_features=2048, out_features=512, bias=False)
        (v_proj): Linear(in_features=2048, out_features=512, bias=False)
        (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
      )
      (mlp): LlamaMLP(
        (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
        (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
        (act_fn): SiLUActivation()
      )
      (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
    )
  )
  (norm): LlamaRMSNorm((2048,), eps=1e-05)
  (rotary_emb): LlamaRotaryEmbedding()
)

In [41]:
# There are 32 transformer blocks or layers. You can access any particular block

llama_model.model.layers[15]

LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear(in_features=2048, out_features=512, bias=False)
    (v_proj): Linear(in_features=2048, out_features=512, bias=False)
    (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
    (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
    (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)

In [42]:
llama_model.lm_head

Linear(in_features=2048, out_features=128256, bias=False)

In [43]:
llama_model.device

device(type='mps', index=0)